<a href="https://colab.research.google.com/github/jonathan9970/jonathan9970.github.io/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from flask import Flask, request, render_template_string
import cv2
import numpy as np
import os
from datetime import datetime

app = Flask(__name__)

SAVE_FOLDER = "forest_images"
os.makedirs(SAVE_FOLDER, exist_ok=True)

last_command = "STOP"

HTML_PAGE = """
<!DOCTYPE html>
<html>
<head>
    <title>Deforestation Robot Controller</title>
    <style>
        body {
            font-family: Arial;
            text-align: center;
            background-color: #eef5ee;
        }
        button {
            width: 120px;
            height: 60px;
            font-size: 18px;
            margin: 10px;
            border-radius: 10px;
        }
        .stop {
            background-color: red;
            color: white;
        }
    </style>
</head>
<body>
    <h1>Deforestation Detection Robot</h1>
    <h2>Human Control Panel</h2>

    <form action="/control" method="post">
        <button name="command" value="FORWARD">Forward</button><br>
        <button name="command" value="LEFT">Left</button>
        <button class="stop" name="command" value="STOP">Stop</button>
        <button name="command" value="RIGHT">Right</button><br>
        <button name="command" value="BACKWARD">Backward</button>
    </form>

    <h3>Current Command: {{ command }}</h3>
</body>
</html>
"""

@app.route("/")
def home():
    return render_template_string(HTML_PAGE, command=last_command)

@app.route("/control", methods=["POST"])
def control_robot():
    global last_command

    last_command = request.form["command"]
    print(f"[CONTROL] Robot command: {last_command}")

    return render_template_string(HTML_PAGE, command=last_command)

@app.route("/get_command", methods=["GET"])
def get_command():
    return last_command

@app.route("/upload", methods=["POST"])
def upload_image():
    file = request.files["image"]

    image_bytes = file.read()
    npimg = np.frombuffer(image_bytes, np.uint8)
    img = cv2.imdecode(npimg, cv2.IMREAD_COLOR)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{SAVE_FOLDER}/forest_{timestamp}.jpg"
    cv2.imwrite(filename, img)

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    lower_green = np.array([35, 40, 40])
    upper_green = np.array([85, 255, 255])

    mask = cv2.inRange(hsv, lower_green, upper_green)

    green_pixels = cv2.countNonZero(mask)
    total_pixels = img.shape[0] * img.shape[1]
    green_percentage = (green_pixels / total_pixels) * 100

    print(f"[IMAGE SAVED] {filename}")
    print(f"[GREEN COVERAGE] {green_percentage:.2f}%")

    if green_percentage < 30:
        print("[ALERT] Possible deforestation detected!")
        result = "Possible deforestation detected"
    else:
        result = "Forest coverage looks normal"

    return {
        "message": "Image received",
        "green_percentage": green_percentage,
        "result": result
    }

if __name__ == "__main__":
    print("Starting robot control server...")
    print("Open this in your browser: http://127.0.0.1:5000")
    app.run(host="0.0.0.0", port=5000)